# 📊 Strongbox Parser - Google Colab Edition

**Convert Strongbox Excel files to Audit Sight format - No installation required!**

## 🚀 Instructions:
1. **Click Runtime → Run all** (or press Ctrl+F9)
2. **Wait for setup** to complete (about 30 seconds)
3. **Upload your file** when prompted
4. **Download** the processed result

**Features:**
- ✅ Processes multiple TXN-FY sheets with real transaction data
- ✅ Auto-detects date ranges from TB data
- ✅ Creates Comparative Trial Balance from your accounts
- ✅ Generates Journal Entries & Lines from your transactions
- ✅ Professional Excel formatting
- ✅ Data cleaning and Unicode handling
- ✅ Complete with all required tabs (Instructions, Banking, etc.)

**Team-friendly: Share this link with anyone who needs to process Strongbox files!**

In [ ]:
#@title 🔧 Setup (Run this first)
print('🔧 Installing required packages...')
!pip install -q openpyxl python-dateutil xlsxwriter
print('✅ Setup complete! Ready to process files.')

In [ ]:
#@title 📚 Import Libraries
import pandas as pd
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import calendar
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from copy import copy
import math
import sys
from google.colab import files
import io
import re
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

In [ ]:
#@title 🏗️ Load Strongbox Parser
class StrongboxParserColab:
    def __init__(self):
        self.source_file = None
        self.start_date = None
        self.end_date = None
        self.start_date = None
        self.source_data = {}
        self.template_data = {}
        self.output_filename = None
        self.date_columns = {}
        self.non_usd_transactions = []  # Store non-USD transactions
        self.non_usd_headers = ['Journal ID', 'Type', 'Journal Entry Description', 'Posted Date', 'Account ID', 'Journal Line Description', 'Name', 'Debit Amount', 'Credit Amount', 'Transaction Currency']  # Headers for non-USD transactions tab

    def print_and_log(self, message):
        print(message)

    def update_status(self, message, progress=None):
        if progress:
            print(f'[{progress:2d}%] {message}')
        else:
            print(message)

    def determine_date_range(self):
        """Determine date range from TB sheet efficiently"""
        self.update_status('Determining date range from TB sheet...', 10)

        # Read the date row with optimized settings
        try:
            date_row = pd.read_excel(
                self.source_file,
                sheet_name='TB',
                header=None,
                nrows=1,
                skiprows=3
                # Removed pyarrow dependency for better compatibility
            )
            date_row = date_row.iloc[0]

            # Convert all values to datetime efficiently using vectorized operations
            dates = pd.to_datetime(date_row, errors='coerce')
            valid_dates = dates[dates.notna()]

            if valid_dates.empty:
                raise Exception('No valid dates found in TB sheet row 4')
            if len(valid_dates) < 2:
                raise Exception('Need at least two dates in TB sheet')

            # Create date_columns mapping
            date_columns = {date: idx for idx, date in enumerate(dates) if pd.notna(date)}

            # Set date range
            self.begin_balance_date = valid_dates.min()
            self.start_date = self.begin_balance_date + relativedelta(days=1)
            self.end_date = valid_dates.max()
            self.date_columns = date_columns

            self.print_and_log(f'📅 Date range: {self.start_date.strftime("%Y-%m-%d")} to {self.end_date.strftime("%Y-%m-%d")}')
            self.print_and_log(f'📅 Found {len(date_columns)} valid dates in TB sheet')



            return date_columns

        except Exception as e:
            self.print_and_log(f'❌ Error determining date range: {str(e)}')
            raise



    def load_source_data(self):
        """Load all required data from the Excel file efficiently using openpyxl directly"""
        self.update_status('Loading transaction data...', 20)

        try:
            # Load workbook once with read-only mode and data_only for better performance
            workbook = openpyxl.load_workbook(self.source_file, read_only=True, data_only=True)

            # Validate required tabs
            available_sheets = workbook.sheetnames
            txn_sheets = [s for s in available_sheets if s.startswith('TXN-FY')]

            # Check for missing required tabs
            missing_tabs = []
            if 'TB' not in available_sheets:
                missing_tabs.append('TB')
            if 'TB-DATA' not in available_sheets:
                missing_tabs.append('TB-DATA')
            if not txn_sheets:
                missing_tabs.append('TXN-FY* (at least one transaction sheet starting with TXN-FY)')

            if missing_tabs:
                if len(missing_tabs) == 1:
                    error_message = f"Unable to parse this Strongbox file. Missing required tab: {missing_tabs[0]}"
                else:
                    tabs_list = ', '.join(missing_tabs[:-1]) + f' and {missing_tabs[-1]}'
                    error_message = f"Unable to parse this Strongbox file. Missing required tabs: {tabs_list}"
                self.print_and_log(f"❌ ERROR: {error_message}")
                self.print_and_log(f"📋 Available sheets in file: {available_sheets}")
                self.update_status("Validation failed - missing required tabs", 0)
                raise Exception(error_message)

            self.print_and_log("✅ All required tabs found:")
            self.print_and_log("  • TB sheet: Found")
            self.print_and_log("  • TB-DATA sheet: Found")
            self.print_and_log(f"  • Transaction sheets: Found {len(txn_sheets)} sheets ({', '.join(txn_sheets)})")

            # Process TXN sheets using pandas for better performance with large datasets
            for sheet_name in txn_sheets:
                try:
                    # Read the sheet with pandas, forcing account-related columns to be strings
                    # to preserve exact format (prevent 1016 -> 1016.0 conversion)
                    dtype_dict = {
                        'Account Id': str,
                        'Account Number/Code': str, 
                        'Account Number': str,
                        'Account Code': str
                    }
                    df = pd.read_excel(self.source_file, sheet_name=sheet_name, dtype=dtype_dict)
                    
                    # Fix account columns to preserve exact decimal formatting (3041.3 should stay 3041.3, not 3041.30)
                    # Remove trailing zeros from decimal account numbers
                    def fix_decimal_formatting(value):
                        if value is None:
                            return ''
                        str_value = str(value).strip()
                        if str_value == '' or str_value.lower() == 'nan':
                            return ''
                        
                        # If it contains a decimal point, remove trailing zeros
                        if '.' in str_value:
                            # Remove trailing zeros after decimal point
                            str_value = str_value.rstrip('0').rstrip('.')
                        
                        return str_value
                    
                    if not df.empty:
                        # Apply decimal formatting fix to account columns
                        account_columns = ['Account Id', 'Account Number/Code', 'Account Number', 'Account Code']
                        for col in account_columns:
                            if col in df.columns:
                                df[col] = df[col].apply(fix_decimal_formatting)
                    
                    # Filter by date range if needed
                    if self.start_date is not None and self.end_date is not None and 'Fiscal Month' in df.columns:
                        # Only include transactions from start_date onwards (not from begin_balance_date)
                        df = df[(df['Fiscal Month'] >= self.start_date) & (df['Fiscal Month'] <= self.end_date)]
                    
                    if not df.empty:
                        # Clean and convert data types
                        df['Transaction Id'] = df['Transaction Id'].astype(str)
                        # Account Id is already a string from dtype specification
                        df['Memo'] = df['Memo'].fillna('')
                        df['Doc/Ref No'] = df['Doc/Ref No'].fillna('')
                        df['Transaction Type'] = df['Transaction Type'].fillna('')
                        df['Relationship Name'] = df['Relationship Name'].fillna('')
                        df['Debit'] = pd.to_numeric(df['Debit'], errors='coerce').fillna(0)
                        df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce').fillna(0)
                        
                        # Store the processed data
                        self.source_data[sheet_name] = df
                        self.print_and_log(f"✅ {sheet_name}: {len(df)} transactions loaded")
                    else:
                        self.print_and_log(f"⚠️ {sheet_name}: No transactions in date range")

                except Exception as e:
                    self.print_and_log(f"⚠️ Error loading {sheet_name}: {str(e)}")

            # Load TB-DATA sheet
            self.print_and_log("\nLoading TB-DATA sheet...")
            self.update_status("Loading TB-DATA sheet...", 25)
            try:
                tb_data = pd.read_excel(self.source_file, sheet_name='TB-DATA')
                self.source_data['TB-DATA'] = tb_data
                self.print_and_log(f"✅ Successfully loaded TB-DATA sheet with {len(tb_data)} rows")

            except Exception as e:
                self.print_and_log(f"⚠️ Warning: Could not load TB-DATA sheet: {str(e)}")
                self.print_and_log("Will use default balance values (0) if TB-DATA is not available")
                self.source_data['TB-DATA'] = None

            workbook.close()
            
            # Log summary of non-USD transactions
            if self.non_usd_transactions:
                self.print_and_log(f"\n⚠️ Non-USD transactions found. These are printed on the 'Non-USD Transactions' tab of the output file")
            else:
                self.print_and_log("\n✅ All transactions are in USD")

            # Load trial balance data using openpyxl
            tb_data = self.load_trial_balance_data()
            self.source_data['TB'] = tb_data

        except Exception as e:
            self.print_and_log(f"❌ Error loading Excel file: {str(e)}")
            raise



    def load_trial_balance_data(self):
        # This method keeps the existing TB sheet loading logic
        self.update_status('Loading trial balance data...', 40)

        # Get the date_columns that were determined in determine_date_range
        date_columns = self.date_columns

        # Find the closest TB dates to our calculated range
        available_tb_dates = sorted(date_columns.keys())

        # Find closest beginning date
        closest_begin_date = None
        for tb_date in available_tb_dates:
            if tb_date <= self.begin_balance_date:
                closest_begin_date = tb_date
            else:
                break

        if closest_begin_date is None:
            closest_begin_date = available_tb_dates[0]

        # Find closest ending date
        closest_end_date = None
        for tb_date in reversed(available_tb_dates):
            if tb_date >= self.end_date:
                closest_end_date = tb_date
            else:
                break

        if closest_end_date is None:
            closest_end_date = available_tb_dates[-1]

        return self._extract_trial_balance_data(date_columns, closest_begin_date, closest_end_date)

    def _extract_trial_balance_data(self, date_columns, closest_begin_date, closest_end_date):
        """Extract data from the TB sheet using openpyxl"""
        try:
            wb = openpyxl.load_workbook(self.source_file, data_only=True, read_only=True)
            tb_sheet = wb['TB']

            # Find Financial Statement Classification Path column
            fin_statement_col = self._find_financial_classification_column(tb_sheet)

            data = []

            # Process rows starting from row 5
            for row_idx in range(5, tb_sheet.max_row + 1):
                try:
                    # Get Account Number/Code from column 5, fallback to Account Name from column 6
                    account_number_code = tb_sheet.cell(row=row_idx, column=5).value
                    account_name = tb_sheet.cell(row=row_idx, column=6).value
                    
                    # Get the original Account ID from column 4 (needed for TB-DATA lookup)
                    original_account_id = tb_sheet.cell(row=row_idx, column=4).value
                    
                    # Use Account Number/Code if available, otherwise use Account Name for DISPLAY
                    # Preserve exact format from input file - don't modify the values
                    if account_number_code is not None and str(account_number_code).strip() and str(account_number_code).strip() != 'None':
                        display_account_id = str(account_number_code).strip()
                    elif account_name is not None and str(account_name).strip():
                        display_account_id = str(account_name).strip()
                    else:
                        # Skip rows with no account identifier
                        continue
                    
                    # Skip if original Account ID is also missing (needed for TB-DATA lookup)
                    if original_account_id is None:
                        continue
                    
                    original_account_id = str(original_account_id).strip()
                    
                    # Only skip rows with exact header matches, including the new header format
                    if display_account_id.lower() in ['account id', 'account', 'account number/code']:
                        continue
                    
                    fin_statement_class = tb_sheet.cell(row=row_idx, column=fin_statement_col).value

                    if fin_statement_class is None:
                        fin_statement_class = ''
                    else:
                        fin_statement_class = str(fin_statement_class).strip()
                    
                    data.append({
                        'Account Id': display_account_id,  # For display in output
                        'Original Account Id': original_account_id,  # For TB-DATA lookup
                        'Account Name': str(account_name).strip() if account_name is not None else '',
                        'Beginning Balance': 0.0,  # Will be populated from TB-DATA in trial balance creation
                        'Ending Balance': 0.0,     # Will be populated from TB-DATA in trial balance creation
                        'Financial Statement Classification': fin_statement_class
                    })
                except Exception as e:
                    continue

            tb_data = pd.DataFrame(data)
            print(f"✅ Extracted {len(data)} account records from TB sheet")
            
            return tb_data

        except Exception as e:
            self.print_and_log(f'Error in _extract_trial_balance_data: {str(e)}')
            raise
        finally:
            try:
                if 'wb' in locals():
                    wb.close()
            except:
                pass

    def _find_financial_classification_column(self, tb_sheet):
        """Find the Financial Statement Classification column in the TB sheet"""
        fin_statement_col = None

        for row_idx in range(1, 6):
            for col_idx in range(1, 15):
                try:
                    cell_value = tb_sheet.cell(row=row_idx, column=col_idx).value
                    if cell_value:
                        cell_text = str(cell_value).lower()
                        if 'financial statement classification' in cell_text:
                            fin_statement_col = col_idx
                            return fin_statement_col
                except Exception:
                    continue

        if not fin_statement_col:
            fin_statement_col = 3  # Default to column C

        return fin_statement_col

    def determine_account_type(self, fs_classification):
        """Determine Account Type based on Financial Statement Classification Path"""
        if pd.isna(fs_classification) or fs_classification == '':
            return ''

        fs_classification = str(fs_classification).strip()

        if fs_classification.startswith('Total Assets'):
            return 'Assets'
        elif fs_classification.startswith('Total Liabilities and Equity → Total Liabilities'):
            return 'Liabilities'
        elif fs_classification.startswith('Total Liabilities and Equity → Total Equity'):
            return 'Equity'
        elif fs_classification.startswith('Net Income → Operating Profit → Gross Profit → Total Net Sales'):
            return 'Income'
        elif fs_classification.startswith('Net Income → Operating Profit → Gross Profit → Total COGS/COS'):
            return 'Expense'
        elif fs_classification.startswith('Net Income → Operating Profit → Total Operating Expenses'):
            return 'Expense'
        else:
            return ''

    def create_trial_balance(self):
        """Create Comparative Trial Balances tab"""
        self.update_status("Creating trial balance...", 70)
        tb_data = self.source_data['TB']
        tb_data_data = self.source_data.get('TB-DATA')

        # Only filter out exact header matches, not all rows containing "account"
        if not tb_data.empty and 'Account Id' in tb_data.columns:
            # Convert to string first to handle any non-string values
            tb_data['Account Id'] = tb_data['Account Id'].astype(str)
            tb_data = tb_data[~((tb_data['Account Id'].str.lower() == "account id") | 
                               (tb_data['Account Id'].str.lower() == "account"))]
        else:
            print("⚠️ Warning: TB data is empty or missing 'Account Id' column")
            print(f"TB data columns: {tb_data.columns.tolist() if not tb_data.empty else 'DataFrame is empty'}")
            print(f"TB data shape: {tb_data.shape}")
            if not tb_data.empty:
                print("First few rows of TB data:")
                print(tb_data.head())

        # Get the earliest and latest TB dates from the TB sheet's date row
        tb_dates = sorted(self.date_columns.keys())
        begin_tb_date = tb_dates[0]
        end_tb_date = tb_dates[-1]

        # Build a mapping from (Account Id, Fiscal Month) to Ending Account Balance
        tbdata_lookup = {}
        if tb_data_data is not None:
            for _, row in tb_data_data.iterrows():
                acc_id = str(row['Account Id']) if 'Account Id' in row else str(row[1])
                fiscal_month = pd.to_datetime(row['Fiscal Month']) if pd.notna(row['Fiscal Month']) else None
                if fiscal_month is not None:
                    key = (acc_id, fiscal_month)
                    bal = row['Ending Account Balance'] if 'Ending Account Balance' in row else row.iloc[6]
                    try:
                        bal = float(bal) if pd.notna(bal) else 0.0
                    except (ValueError, TypeError):
                        bal = 0.0
                    tbdata_lookup[key] = bal

        begin_balances = []
        end_balances = []
        if not tb_data.empty and 'Account Id' in tb_data.columns and 'Original Account Id' in tb_data.columns:
            for i, (display_id, original_id) in enumerate(zip(tb_data['Account Id'], tb_data['Original Account Id'])):
                original_id_str = str(original_id)
                begin_bal = tbdata_lookup.get((original_id_str, begin_tb_date), 0.0)
                end_bal = tbdata_lookup.get((original_id_str, end_tb_date), 0.0)
                begin_balances.append(begin_bal)
                end_balances.append(end_bal)
        else:
            print("⚠️ Warning: Cannot process balances - TB data is empty or missing required columns")

        if not tb_data.empty and 'Account Id' in tb_data.columns:
            trial_balance = pd.DataFrame({
                'Account ID': tb_data['Account Id'],  # This now contains Account Number/Code or Account Name
                'Account Name': tb_data['Account Name'] if 'Account Name' in tb_data.columns else [''] * len(tb_data),
                'Beginning Balance \n(Prior Period Balance)': begin_balances,
                'Ending Balance': end_balances,
                'Account Type \n(see Mapping Categories tab)': [''] * len(tb_data),
                'Account Mapping \n(see Mapping Categories tab)': [''] * len(tb_data),
                'Account Description': tb_data['Financial Statement Classification'] if 'Financial Statement Classification' in tb_data.columns else [''] * len(tb_data)
            })
        else:
            # Create empty trial balance if no data
            print("⚠️ Creating empty trial balance due to missing data")
            trial_balance = pd.DataFrame({
                'Account ID': [],
                'Account Name': [],
                'Beginning Balance \n(Prior Period Balance)': [],
                'Ending Balance': [],
                'Account Type \n(see Mapping Categories tab)': [],
                'Account Mapping \n(see Mapping Categories tab)': [],
                'Account Description': []
            })

        # Apply the class method to populate the Account Type column
        trial_balance['Account Type \n(see Mapping Categories tab)'] = trial_balance['Account Description'].apply(self.determine_account_type)

        # Count how many accounts were classified for each type
        account_type_counts = trial_balance['Account Type \n(see Mapping Categories tab)'].value_counts()
        self.print_and_log("\nAccount Type classification summary:")
        for account_type, count in account_type_counts.items():
            if account_type != '':
                self.print_and_log(f"  {account_type}: {count} accounts")

        account_type_col = 'Account Type \n(see Mapping Categories tab)'
        unclassified_count = (trial_balance[account_type_col] == '').sum()
        self.print_and_log(f"  Unclassified: {unclassified_count} accounts")



        # Only remove rows that are definitely headers (contain exactly "Account ID")
        headers_to_remove = []
        for idx, row in trial_balance.iterrows():
            account_id = str(row['Account ID']).lower() if pd.notna(row['Account ID']) else ""
            if account_id == "account id" or account_id == "account":
                headers_to_remove.append(idx)
                self.print_and_log(f"Removing header row: {row['Account ID']}")

        if headers_to_remove:
            trial_balance = trial_balance.drop(headers_to_remove)

        # Verify there are no empty account IDs but don't filter out other accounts
        trial_balance = trial_balance[trial_balance['Account ID'].notna() & (trial_balance['Account ID'] != '')]

        # Reset the index after filtering
        trial_balance = trial_balance.reset_index(drop=True)

        return trial_balance

    def create_journal_entries(self):
        """Create Journal Entries & Lines tabs"""
        self.update_status('Creating journal entries...', 60)

        # Get transaction sheets
        transaction_sheets = {k: v for k, v in self.source_data.items() if k.startswith('TXN-FY')}

        processed_sheets = {}  # Changed from list to dictionary
        for sheet_name, df in transaction_sheets.items():
            try:
                df_copy = df.copy()

                # Convert columns to appropriate types
                df_copy['Transaction Id'] = df_copy['Transaction Id'].astype(str)
                df_copy['Memo'] = df_copy['Memo'].fillna('')
                df_copy['Doc/Ref No'] = df_copy['Doc/Ref No'].fillna('')
                # Account Id is already a string from dtype specification

                # Handle optional columns
                for col in ['Transaction Type', 'Relationship Name']:
                    if col in df_copy.columns:
                        df_copy[col] = df_copy[col].fillna('')
                    else:
                        df_copy[col] = ''

                # Convert numeric columns
                df_copy['Debit'] = pd.to_numeric(df_copy['Debit'], errors='coerce').fillna(0)
                df_copy['Credit'] = pd.to_numeric(df_copy['Credit'], errors='coerce').fillna(0)

                # Create the required columns
                # Determine Account ID to use: Account Number/Code if available, otherwise Account Name
                # Preserve exact format from input file - don't modify the values
                account_ids = []
                for _, row in df_copy.iterrows():
                    account_number_code = None
                    account_name = None
                    
                    # Check if Account Number/Code column exists and get its value
                    if 'Account Number/Code' in df_copy.columns:
                        account_number_code = row.get('Account Number/Code')
                    elif 'Account Number' in df_copy.columns:
                        account_number_code = row.get('Account Number')
                    elif 'Account Code' in df_copy.columns:
                        account_number_code = row.get('Account Code')
                    
                    # Get Account Name as fallback
                    if 'Account Name' in df_copy.columns:
                        account_name = row.get('Account Name')
                    
                    # Use Account Number/Code if available and not blank, otherwise use Account Name
                    # Preserve original format - just convert to string and check for validity
                    if account_number_code is not None and str(account_number_code).strip() and str(account_number_code).strip().lower() != 'nan':
                        account_ids.append(str(account_number_code).strip())
                    elif account_name is not None and str(account_name).strip() and str(account_name).strip().lower() != 'nan':
                        account_ids.append(str(account_name).strip())
                    else:
                        # Fallback to original Account Id if neither is available, but avoid 'nan'
                        original_account_id = str(row.get('Account Id', '')).strip()
                        if original_account_id and original_account_id.lower() != 'nan':
                            account_ids.append(original_account_id)
                        else:
                            account_ids.append('')  # Use empty string instead of 'nan'
                
                processed_df = pd.DataFrame({
                    'Journal ID': df_copy['Transaction Id'],
                    'Type': df_copy['Transaction Type'],
                    'Journal Entry Description': df_copy['Doc/Ref No'],
                    'Posted Date': df_copy['Transaction Date'],
                    'Account ID': account_ids,
                    'Journal Line Description': df_copy['Memo'],
                    'Name': df_copy['Relationship Name'] if 'Relationship Name' in df_copy.columns else '',
                    'Debit Amount': df_copy['Debit'],
                    'Credit Amount': df_copy['Credit']
                })

                processed_sheets[sheet_name] = processed_df  # Store in dictionary with sheet name as key
                self.print_and_log(f'✅ Successfully processed sheet: {sheet_name}')

            except Exception as e:
                self.print_and_log(f'⚠️ Error processing sheet {sheet_name}: {str(e)}')
                continue

        if not processed_sheets:
            raise Exception('No sheets were successfully processed')

        self.print_and_log(f'✅ Successfully processed {len(processed_sheets)} sheets')
        return processed_sheets

    def create_output_file(self):
        """Create Excel output file with professional formatting"""
        self.update_status('Creating Excel output...', 80)

        start_str = self.start_date.strftime('%Y%m%d')
        end_str = self.end_date.strftime('%Y%m%d')
        self.output_filename = f'Processed_Strongbox_{start_str}_{end_str}.xlsx'

        # Create trial balance and journal entries
        trial_balance = self.create_trial_balance()
        journal_entries_dict = self.create_journal_entries()

        # Clean data for Excel
        trial_balance = self._clean_data_for_excel(trial_balance)
        for sheet_name in journal_entries_dict:
            journal_entries_dict[sheet_name] = self._clean_data_for_excel(journal_entries_dict[sheet_name])

        # Create Excel workbook with proper styling
        workbook, styles = self._create_excel_workbook()

        # Create main data sheets first (in desired order)
        tb_sheet = self._create_trial_balance_sheet(workbook, trial_balance, styles)
        je_sheets = self._create_journal_entries_sheets(workbook, journal_entries_dict, styles)
        
        # Create Non-USD Transactions sheet if needed
        if self.non_usd_transactions:
            self._create_non_usd_transactions_sheet(workbook, styles)

        # Create other sheets after main data sheets
        self._create_other_sheets(workbook, styles)

        # Save workbook
        workbook.save(self.output_filename)
        workbook.close()

        self.print_and_log(f'✅ Output file created: {self.output_filename}')
        return self.output_filename

    def _create_excel_workbook(self):
        """Create Excel workbook with basic styling setup"""
        workbook = openpyxl.Workbook()

        # Define styles
        from openpyxl.styles import Font, PatternFill, Alignment

        header_font = Font(name='Arial', size=12, bold=True, color='FFFFFF')
        blue_fill = PatternFill(start_color='0070C0', end_color='0070C0', fill_type='solid')
        gray_fill = PatternFill(start_color='999999', end_color='999999', fill_type='solid')
        dark_blue_fill = PatternFill(start_color='002060', end_color='002060', fill_type='solid')
        center_alignment = Alignment(horizontal='center', vertical='center')

        styles = {
            'header_font': header_font,
            'blue_fill': blue_fill,
            'gray_fill': gray_fill,
            'dark_blue_fill': dark_blue_fill,
            'center_alignment': center_alignment
        }

        # Remove default sheet
        if 'Sheet' in workbook.sheetnames:
            workbook.remove(workbook['Sheet'])

        return workbook, styles

    def _create_trial_balance_sheet(self, workbook, trial_balance, styles):
        """Create and format the Comparative Trial Balances sheet"""
        # Create trial balance sheet
        tb_sheet = workbook.create_sheet('Comparative Trial Balances')

        # Set column widths for trial balance (converting px to Excel units)
        tb_sheet.column_dimensions['A'].width = 14.3  # 100px
        tb_sheet.column_dimensions['B'].width = 34.3  # 240px
        tb_sheet.column_dimensions['C'].width = 15.7  # 110px
        tb_sheet.column_dimensions['D'].width = 15.7  # 110px
        tb_sheet.column_dimensions['E'].width = 15.7  # 110px
        tb_sheet.column_dimensions['F'].width = 34.3  # 240px
        tb_sheet.column_dimensions['G'].width = 34.3  # 240px

        # Write headers
        tb_sheet.append(['Required'] * 5 + ['Optional'] * 2)
        tb_sheet.append(list(trial_balance.columns))

        # Style row 1 headers (Required/Optional)
        for col in range(1, 6):  # Columns A-E
            cell = tb_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']

        for col in range(6, 8):  # Columns F-G
            cell = tb_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['gray_fill']
            cell.alignment = styles['center_alignment']

        # Style row 2 headers (column names) and set row height
        tb_sheet.row_dimensions[2].height = 25  # 33px ≈ 25 points
        for col in range(1, 8):  # All columns A-G
            cell = tb_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']

        # Write trial balance data with error handling
        for _, row in trial_balance.iterrows():
            row_values = []
            for col in trial_balance.columns:
                value = row[col]
                try:
                    # Double-check the value is clean
                    if isinstance(value, str) and len(value) > 1000:
                        value = value[:1000]
                    row_values.append(value)
                except:
                    row_values.append("ERROR")
            tb_sheet.append(row_values)

        return tb_sheet

    def _create_journal_entries_sheets(self, workbook, journal_entries_dict, styles):
        """Create and format multiple Journal Entries & Lines sheets, one for each TXN sheet"""
        # Create journal entries sheets
        created_sheets = []

        # Sort the sheets by name to ensure consistent ordering
        sheet_names = sorted(journal_entries_dict.keys())

        for idx, sheet_name in enumerate(sheet_names, 1):
            journal_entries = journal_entries_dict[sheet_name]
            sheet_title = f'Journal Entries & Lines {idx}'

            # Create the sheet
            je_sheet = workbook.create_sheet(sheet_title)

            # Set column widths for journal entries (converting px to Excel units)
            je_sheet.column_dimensions['A'].width = 14.3  # 100px
            je_sheet.column_dimensions['B'].width = 48.6  # 340px
            je_sheet.column_dimensions['C'].width = 15.7  # 110px
            je_sheet.column_dimensions['D'].width = 20.0  # 140px
            je_sheet.column_dimensions['E'].width = 35.7  # 250px
            je_sheet.column_dimensions['F'].width = 15.7  # 110px
            je_sheet.column_dimensions['G'].width = 15.7  # 110px
            je_sheet.column_dimensions['H'].width = 15.7  # 110px
            je_sheet.column_dimensions['I'].width = 15.7  # 110px

            # Write headers
            je_sheet.append(['Required', 'Optional', 'Optional', 'Required', 'Required', 'Optional', 'Optional', 'Required', 'Required'])
            je_sheet.append(list(journal_entries.columns))

            # Style row 1 headers (Required/Optional)
            required_cols = [1, 4, 5, 8, 9]  # Columns A, D, E, H, I
            optional_cols = [2, 3, 6, 7]  # Columns B, C, F, G

            for col in required_cols:
                cell = je_sheet.cell(row=1, column=col)
                cell.font = styles['header_font']
                cell.fill = styles['blue_fill']
                cell.alignment = styles['center_alignment']

            for col in optional_cols:
                cell = je_sheet.cell(row=1, column=col)
                cell.font = styles['header_font']
                cell.fill = styles['gray_fill']
                cell.alignment = styles['center_alignment']

            # Style row 2 headers (column names)
            for col in range(1, 10):  # All columns
                cell = je_sheet.cell(row=2, column=col)
                cell.font = styles['header_font']
                cell.fill = styles['dark_blue_fill']
                cell.alignment = styles['center_alignment']

            # Write journal entries data with error handling
            for _, row in journal_entries.iterrows():
                row_values = []
                for col in journal_entries.columns:
                    value = row[col]
                    try:
                        # Double-check the value is clean
                        if isinstance(value, str) and len(value) > 1000:
                            value = value[:1000]
                        row_values.append(value)
                    except:
                        row_values.append("ERROR")
                je_sheet.append(row_values)

            # Apply date formatting to column D (Posted Date) in Journal Entries & Lines
            for row_num in range(3, je_sheet.max_row + 1):  # Start from row 3 (after headers)
                cell = je_sheet.cell(row=row_num, column=4)
                cell.number_format = 'M/D/YYYY'

            created_sheets.append(je_sheet)
            self.print_and_log(f'✅ Created sheet "{sheet_title}" from {sheet_name}')

        return created_sheets

    def _clean_data_for_excel(self, df):
        """Clean data for Excel output"""
        def ultra_clean_value(value):
            """Clean individual values for Excel"""
            if pd.isna(value):
                return ''

            # Convert to string and clean
            str_value = str(value)

            # Remove or replace problematic characters
            str_value = str_value.replace('\x00', '')  # Null bytes
            str_value = str_value.replace('\r', ' ')   # Carriage returns
            str_value = str_value.replace('\n', ' ')   # Line feeds
            str_value = str_value.replace('\t', ' ')   # Tabs

            # Handle other potential issues
            if len(str_value) > 32767:  # Excel cell character limit
                str_value = str_value[:32767]

            return str_value

        # Create a copy to avoid modifying the original
        df_clean = df.copy()

        # Clean all string columns
        for col in df_clean.columns:
            if df_clean[col].dtype == object:  # Only clean string/object columns
                df_clean[col] = df_clean[col].apply(ultra_clean_value)

        return df_clean

    def _create_non_usd_transactions_sheet(self, workbook, styles):
        """Create Non-USD Transactions sheet with Journal Entries columns plus Transaction Currency"""
        non_usd_sheet = workbook.create_sheet('Non-USD Transactions')
        
        if not self.non_usd_transactions:
            return non_usd_sheet
        
        # Write headers (Journal Entries & Lines columns + Transaction Currency)
        non_usd_sheet.append(self.non_usd_headers)
        
        # Style headers
        for col_idx in range(1, len(self.non_usd_headers) + 1):
            cell = non_usd_sheet.cell(row=1, column=col_idx)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']
        
        # Write non-USD transaction data
        for transaction in self.non_usd_transactions:
            row_data = []
            for header in self.non_usd_headers:
                value = transaction.get(header, '')
                # Clean the value for Excel
                if isinstance(value, str) and len(value) > 1000:
                    value = value[:1000]
                row_data.append(value)
            non_usd_sheet.append(row_data)
        
        # Set column widths (optimized for Journal Entries format)
        column_widths = {
            'A': 14.3,  # Journal ID
            'B': 48.6,  # Type
            'C': 15.7,  # Journal Entry Description
            'D': 20.0,  # Posted Date
            'E': 35.7,  # Account ID
            'F': 15.7,  # Journal Line Description
            'G': 15.7,  # Name
            'H': 15.7,  # Debit Amount
            'I': 15.7,  # Credit Amount
            'J': 15.7   # Transaction Currency
        }
        
        for col_letter, width in column_widths.items():
            non_usd_sheet.column_dimensions[col_letter].width = width
        
        # Apply date formatting to Posted Date column (column D)
        for row_num in range(2, non_usd_sheet.max_row + 1):
            cell = non_usd_sheet.cell(row=row_num, column=4)
            cell.number_format = 'M/D/YYYY'
        
        self.print_and_log(f'✅ Created Non-USD Transactions sheet with {len(self.non_usd_transactions)} transactions')
        return non_usd_sheet

    def _create_other_sheets(self, workbook, styles):
        """Create and format all other sheets in the specified order"""
        # Create Instructions sheet
        instructions_sheet = workbook.create_sheet('Instructions')
        instructions_sheet.append(['Content for Instructions'])

        # Create Data Validation Tests sheet
        validation_sheet = workbook.create_sheet('Data Validation Tests')
        validation_sheet.append(['Content for Data Validation Tests'])

        # Create Notes sheet
        notes_sheet = workbook.create_sheet('Notes')
        notes_sheet.append(['Content for Notes'])

        # Create Banking Accts sheet with specific formatting
        banking_accts_sheet = workbook.create_sheet('Banking Accts')

        # Add headers for Banking Accts
        banking_accts_sheet.append(['Required', 'Required', 'Optional', 'Optional'])
        banking_accts_sheet.append(['Account Number', 'Account Name', 'Institution', 'Currency'])

        # Style row 1 headers for Banking Accts
        for col in range(1, 3):  # Columns A-B (Required)
            cell = banking_accts_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']

        for col in range(3, 5):  # Columns C-D (Optional)
            cell = banking_accts_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['gray_fill']
            cell.alignment = styles['center_alignment']

        # Style row 2 headers for Banking Accts
        for col in range(1, 5):  # All columns A-D
            cell = banking_accts_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']

        # Create Banking Txn sheet with specific formatting
        banking_txn_sheet = workbook.create_sheet('Banking Txn')

        # Add headers for Banking Txn
        banking_txn_sheet.append(['Required', 'Required', 'Required', 'Required'])
        banking_txn_sheet.append(['Posted Date', 'Description', 'Amount', 'Account Number'])

        # Style row 1 headers for Banking Txn (all Required)
        for col in range(1, 5):  # Columns A-D (all Required)
            cell = banking_txn_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']

        # Style row 2 headers for Banking Txn
        for col in range(1, 5):  # All columns A-D
            cell = banking_txn_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']

        # Create Mapping Categories sheet
        mapping_sheet = workbook.create_sheet('Mapping Categories')
        mapping_sheet.append(['Content for Mapping Categories'])

    def run(self):
        try:
            self.print_and_log('🚀 Starting Strongbox Parser...')
            
            # Currency check - GUARANTEED to show up
            self.print_and_log('')
            self.print_and_log('💰 CHECKING PRESENTATION CURRENCY...')
            self.print_and_log('=' * 40)
            
            currency_found = False
            currency_message = ""
            
            try:
                self.print_and_log('📋 Opening file to check TOC tab...')
                wb = openpyxl.load_workbook(self.source_file, data_only=True, read_only=True)
                self.print_and_log(f'📋 Available sheets: {wb.sheetnames}')
                
                if 'TOC' in wb.sheetnames:
                    self.print_and_log('📋 TOC tab found, checking cell C10...')
                    toc_sheet = wb['TOC']
                    currency_cell = toc_sheet['C10'].value
                    self.print_and_log(f'📋 Raw value in C10: "{currency_cell}"')
                    
                    if currency_cell:
                        currency = str(currency_cell).strip()
                        self.print_and_log(f'📋 Cleaned currency value: "{currency}"')
                        
                        if currency.lower() == '(usd) united states dollar':
                            currency_message = '✅ PRESENTATION CURRENCY: USD ✅'
                        else:
                            currency_message = f'⚠️ WARNING: PRESENTATION CURRENCY IS NOT USD! ⚠️\n    Found: "{currency}"'
                        currency_found = True
                    else:
                        currency_message = '⚠️ Cell C10 in TOC tab is empty'
                else:
                    currency_message = '⚠️ TOC tab not found in file'
                    
                wb.close()
                
            except Exception as e:
                currency_message = f'⚠️ Error checking currency: {str(e)}'
            
            # Always show the currency result
            self.print_and_log('')
            self.print_and_log(currency_message)
            self.print_and_log('=' * 40)
            self.print_and_log('')
            
            # Determine date range from TB sheet
            self.date_columns = self.determine_date_range()
            
            # Load source data using the determined date range
            self.load_source_data()
            
            # Create output file
            output_file = self.create_output_file()
            
            self.update_status('Processing complete!', 100)
            self.print_and_log('\n🎉 SUCCESS! Processing completed!')

            return output_file

        except Exception as e:
            self.print_and_log(f'\n❌ Error: {str(e)}')
            raise

print('✅ Complete Strongbox Parser loaded and ready!')


In [ ]:
#@title 📁 Upload Your Strongbox Excel File

print('📁 UPLOAD YOUR STRONGBOX EXCEL FILE')
print('=' * 40)
print('Requirements:')
print('  ✓ Excel file (.xlsx format)')
print('  ✓ TB sheet with trial balance data')
print('  ✓ TB-DATA sheet with balance information')
print('  ✓ TXN-FY sheets with transactions')
print('  ✓ Dates in row 4 of TB sheet')
print()
print('Click Choose Files below and select your file:')

uploaded = files.upload()

if uploaded:
    source_filename = list(uploaded.keys())[0]
    file_size = len(uploaded[source_filename])
    print(f'\n✅ SUCCESS! File uploaded:')
    print(f'   📄 Name: {source_filename}')
    print(f'   📊 Size: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)')
    print(f'\n🔄 Ready to process! Run the next cell.')
else:
    print('\n❌ No file uploaded. Please try again.')
    source_filename = None

In [ ]:
#@title 🔄 Process Your File

if 'source_filename' in globals() and source_filename:
    print('🔄 PROCESSING YOUR STRONGBOX FILE')
    print('=' * 35)
    print(f'File: {source_filename}')
    print()

    # Check if StrongboxParserColab is defined
    try:
        StrongboxParserColab
    except NameError:
        print('❌ ERROR: StrongboxParserColab class not found!')
        print()
        print('🔧 SOLUTION:')
        print('1. Go back to Step 3: "🏗️ Load Complete Strongbox Parser"')
        print('2. Click the ▶️ button to run that cell')
        print('3. Wait for "✅ Complete Strongbox Parser loaded and ready!" message')
        print('4. Then come back and run this cell again')
        print()
        print('💡 TIP: You can also click Runtime → Run all to run all cells in order')
    else:
        # Initialize and run parser
        parser = StrongboxParserColab()
        parser.source_file = source_filename

        try:
            output_file = parser.run()

            print('\n' + '=' * 50)
            print('🎉 SUCCESS! YOUR FILE HAS BEEN PROCESSED!')
            print('=' * 50)
            print(f'✅ Output file: {output_file}')
            print('\n📋 Your file contains:')
            print('   📊 Comparative Trial Balances (Real Data from TB-DATA)')
            print('   📝 Journal Entries & Lines (Your Real Transactions)')
            print('   📋 Professional formatting and data cleaning')
            print('\n📥 Ready for download! Run the next cell.')

        except Exception as e:
            print('\n' + '=' * 40)
            print('❌ PROCESSING ERROR')
            print('=' * 40)
            print(f'Error: {str(e)}')
            print('\n🔍 Please check:')
            print('  • File has TB sheet with trial balance')
            print('  • File has TB-DATA sheet with balances')
            print('  • TB sheet has dates in row 4')
            print('  • File has TXN-FY sheets with transactions')
            print('  • File is not password protected')
            print('  • File format is .xlsx (not .xls)')
            output_file = None

else:
    print('⚠️ Please upload a file first by running the cell above.')
    output_file = None

In [ ]:
#@title 📥 Download Your Processed File

if 'output_file' in globals() and output_file:
    print('📥 DOWNLOADING YOUR PROCESSED FILE')
    print('=' * 38)
    print(f'File: {output_file}')
    print('\nThe file will download to your browser\'s Downloads folder.')
    print()

    # Check if file exists before downloading
    import os
    if os.path.exists(output_file):
        # Download the file
        files.download(output_file)

        print('✅ Download started!')
        print('\n🎉 ALL DONE!')
        print('=' * 15)
        print('Your Audit Sight formatted file is ready!')
        print('\n💡 What you got:')
        print('  • Complete trial balance with TB-DATA balances')
        print('  • All journal entries from your TXN-FY sheets')
        print('  • Proper date filtering and account classification')
        print('  • Professional Excel formatting')
        print('  • All required tabs for Audit Sight import')
        print('\n📊 Ready for Audit Sight import!')
    else:
        print('❌ Error: Output file not found!')
        print(f'Expected file: {output_file}')

else:
    print('⚠️ No file ready for download.')
    print('Please upload and process a file first.')